# Impulse-Response Experiments

Dedicated paired baseline-versus-shock macro workflow. The implementation lives in `src/experiments/irf.py`; the historical command remains available through `run_irf_experiment.py`.

In [2]:
# Setup
%load_ext autoreload
%autoreload 2

from pathlib import Path

from src.experiments.irf import run_irf_experiment
from src.irf_analysis import plot_irfs

from macromodel.utils.prehooks.irf_shocks import ShockSpec

## Inputs

In [4]:
RUN_IRF = True
COUNTRY = "FRA"
SEEDS = [12, 13, 14, 15, 16, 17, 18, 19, 20]
T_MAX = 50
HORIZON_PERIODS = 4
N_JOBS = 8
OUTPUT_DIR = Path("data/output_data/irf-experiment")

shock_specs = (
    ShockSpec(
        name="gov_cons_10pct",
        kind="government_consumption",
        period=20,
        magnitude=0.10,
        duration=1,
        mode="multiplicative",
    ),
    ShockSpec(name="policy_rate_100bp", kind="policy_rate", period=20, magnitude=0.01, duration=4, mode="additive"),
    ShockSpec(name="income_tax_1pp", kind="income_tax", period=20, magnitude=0.01, duration=4, mode="additive"),
    ShockSpec(name="unemp_2pp", kind="unemployment_rate", period=20, magnitude=0.02, duration=4, mode="additive"),
)

## Run

In [ ]:
outputs = None
if RUN_IRF:
    outputs = run_irf_experiment(
        seeds=SEEDS,
        t_max=T_MAX,
        shock_specs=shock_specs,
        horizon_periods=HORIZON_PERIODS,
        output_dir=OUTPUT_DIR,
        country_iso3=COUNTRY,
        n_jobs=N_JOBS,
    )
else:
    print("Set RUN_IRF = True to run the experiment.")

## Plots

In [ ]:
if outputs is not None:
    for shock in ("policy_rate_100bp", "income_tax_1pp", "unemp_2pp"):
        figure = plot_irfs(
            outputs["analysis_dir"],
            shocks=[shock],
            variables="household_consumption",
            percent=True,
            title=f"{shock} shock on household consumption",
        )
        figure.show()

In [14]:
from macro_data.configuration.countries import Country
from macro_data.processing.country_data.tax_data import TaxData
from macro_data.readers.default_readers import DataPaths
from macro_data.readers.economic_data.eurostat_reader import EuroStatReader
from macro_data.readers.economic_data.oecd_economic_data import OECDEconData
from macro_data.readers.economic_data.world_bank_reader import WorldBankReader

# Select any country and year available in the raw reader data.
TAX_COUNTRY = COUNTRY
TAX_YEAR = 2010

country = Country(TAX_COUNTRY)
raw_data_path = Path("data/raw_data")
paths = DataPaths.default_paths(raw_data_path, [TAX_YEAR])
world_bank = WorldBankReader(paths.world_bank_path)
oecd_econ = OECDEconData(paths.oecd_econ_path, scale_dict={country: 1})
eurostat = EuroStatReader(paths.eurostat_path, paths.country_codes_path)


class TaxReaders:
    world_bank = world_bank
    oecd_econ = oecd_econ
    eurostat = eurostat

    @staticmethod
    def get_export_taxes(country, year):
        return world_bank.get_tau_exp(country, year)


readers = TaxReaders()


def tax_data_from_readers(readers, country, year):
    return TaxData(
        value_added_tax=readers.world_bank.get_tau_vat(country, year),
        export_tax=readers.get_export_taxes(country, year),
        employer_social_insurance_tax=readers.oecd_econ.read_tau_sif(country, year),
        employee_social_insurance_tax=readers.oecd_econ.read_tau_siw(country, year),
        profit_tax=readers.oecd_econ.read_tau_firm(country, year),
        income_tax=readers.oecd_econ.read_tau_income(country, year),
        risk_premium=readers.eurostat.firm_risk_premium(country, year),
        capital_formation_tax=readers.eurostat.taxrate_on_capital_formation(country, year),
    )


tax_data = tax_data_from_readers(readers, country, TAX_YEAR)

tax_rates = {
    "VAT": tax_data.value_added_tax,
    "Export tax": tax_data.export_tax,
    "Employer social insurance": tax_data.employer_social_insurance_tax,
    "Employee social insurance": tax_data.employee_social_insurance_tax,
    "Profit tax": tax_data.profit_tax,
    "Income tax": tax_data.income_tax,
    "Capital formation tax": tax_data.capital_formation_tax,
}

for name, rate in tax_rates.items():
    print(f"{name}: {rate:.6%}")

VAT: 10.722087%
Export tax: 0.000000%
Employer social insurance: 41.730000%
Employee social insurance: 13.700000%
Profit tax: 34.430000%
Income tax: 21.910000%
Capital formation tax: 23.233851%
